# 01 Data audit and experiment check

**Summary.** Before any modelling, this notebook checks what the data contains, when the marketing decision was made and whether the SMS experiment was run fairly. Customers who received the SMS and customers who did not look the same on every piece of information known before the SMS, so the experiment can be trusted. One customer field contained information from after the SMS; it was cut back to what was known at the time of the decision.

**Research question.** Can the X5 data be analysed as a randomized experiment, and what has to be fixed before modelling?

## Setup

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from x5uplift import plotting as P
from x5uplift.audit_utils import srm_chisquare
from x5uplift.config import AGE_MAX, AGE_MIN, CUTOFF, EXPECTED_TREATMENT_SHARE, FEATURES_PATH, SPLITS_PATH
from x5uplift.features import connect
from x5uplift.splits import make_splits

pd.set_option("display.width", 140)
P.set_style()

con = connect()  # DuckDB views: clients, train, purchases, products

def q(sql):
    """Run a SQL query and return the result as a pandas DataFrame."""
    return con.execute(sql).df()

q("""
SELECT 'clients' AS table_name, COUNT(*) AS n_rows FROM clients
UNION ALL SELECT 'train', COUNT(*) FROM train
UNION ALL SELECT 'purchases', COUNT(*) FROM purchases
UNION ALL SELECT 'products', COUNT(*) FROM products
""")

,table_name,n_rows
0,clients,400162
1,train,200039
2,purchases,45786568
3,products,43038


## 1. What the data contains

Four tables: `clients` (one row per loyalty card holder), `train` (the experiment: who received the SMS and whether they bought afterwards), `purchases` (every item bought before the SMS) and `products` (a reference table from the official competition archive). The row counts match the documentation. Client ids are unique and every client in the experiment has a purchase history.

### Purchases are item rows, not baskets

In [3]:
q("""
WITH per_txn AS (
    SELECT
        transaction_id,
        COUNT(*)                                AS n_items,
        COUNT(DISTINCT client_id)               AS n_clients,
        COUNT(DISTINCT transaction_datetime)    AS n_datetimes,
        COUNT(DISTINCT store_id)                AS n_stores,
        COUNT(DISTINCT purchase_sum)            AS n_purchase_sums,
        COUNT(DISTINCT regular_points_received) AS n_points_values,
        MAX(purchase_sum)                       AS purchase_sum,
        SUM(trn_sum_from_iss)                   AS iss_sum,
        SUM(trn_sum_from_iss) + COALESCE(SUM(trn_sum_from_red), 0) AS iss_red_sum
    FROM purchases
    GROUP BY transaction_id
)
SELECT
    COUNT(*)                                                       AS n_transactions,
    AVG(n_items)                                                   AS avg_items_per_txn,
    AVG(CASE WHEN n_clients > 1 THEN 1 ELSE 0 END)                 AS share_multi_client,
    AVG(CASE WHEN n_datetimes > 1 THEN 1 ELSE 0 END)               AS share_multi_datetime,
    AVG(CASE WHEN n_stores > 1 THEN 1 ELSE 0 END)                  AS share_multi_store,
    AVG(CASE WHEN n_purchase_sums > 1 THEN 1 ELSE 0 END)           AS share_varying_purchase_sum,
    AVG(CASE WHEN n_points_values > 1 THEN 1 ELSE 0 END)           AS share_varying_points,
    AVG(CASE WHEN ABS(iss_sum - purchase_sum) <= 1 THEN 1 ELSE 0 END)     AS share_iss_matches_total,
    AVG(CASE WHEN ABS(iss_red_sum - purchase_sum) <= 1 THEN 1 ELSE 0 END) AS share_iss_red_matches_total,
    MEDIAN(iss_red_sum / NULLIF(purchase_sum, 0))                  AS median_items_to_total_ratio
FROM per_txn
""").T

RuntimeError: Query interrupted

Each row is one item. The 8,045,201 transaction ids are baskets with 5.7 items on average. `purchase_sum` and the point columns are basket totals repeated on every item row, so summing them over rows would inflate spend about 5.7 times. 28 transaction ids are shared by two clients (most likely short hash collisions), so a basket is identified by the pair (client, transaction), which gives 8,045,229 baskets. Item amounts add up to the basket total within one unit in 87% of baskets; the rest is unexplained and probably discounts.

**Rules used from here on:** spend is taken once per basket from `purchase_sum`; item amounts are used only for shares, such as the share of spend in a category.

### Product table

In [ ]:
prod_profile = q("""
SELECT
    COUNT(*)                   AS n_rows,
    COUNT(DISTINCT product_id) AS n_products,
    COUNT(DISTINCT level_1)    AS n_level_1,
    COUNT(DISTINCT level_2)    AS n_level_2,
    COUNT(DISTINCT level_3)    AS n_level_3,
    COUNT(DISTINCT level_4)    AS n_level_4,
    COUNT(DISTINCT brand_id)   AS n_brands,
    AVG(CASE WHEN level_1 IS NULL THEN 1 ELSE 0 END)  AS null_level_1,
    AVG(CASE WHEN brand_id IS NULL THEN 1 ELSE 0 END) AS null_brand,
    AVG(is_own_trademark)      AS share_own_trademark,
    AVG(is_alcohol)            AS share_alcohol
FROM products
""")

coverage = q("""
WITH purchased AS (
    SELECT product_id, COUNT(*) AS n_item_rows FROM purchases GROUP BY 1
)
SELECT
    COUNT(*) AS n_purchased_products,
    SUM(CASE WHEN p.product_id IS NULL THEN 1 ELSE 0 END) AS n_missing_in_products
FROM purchased AS u
LEFT JOIN products AS p USING (product_id)
""")

display(prod_profile.T)
display(coverage.T)

43,038 products with a unique id and a four level category tree (3, 42, 201 and 790 categories). All 42,530 purchased products are in the table. Category, brand and vendor ids are hashed, so categories can be used as features but cannot be named.

### Data quality: gender, basket size, quantities

In [ ]:
gender = q("""
SELECT gender, COUNT(*) AS n_clients,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM clients
GROUP BY 1
ORDER BY 1
""")

baskets = q("""
WITH b AS (
    SELECT client_id, transaction_id, MAX(purchase_sum) AS basket_sum, COUNT(*) AS n_items
    FROM purchases
    GROUP BY client_id, transaction_id
)
SELECT
    COUNT(*)                                            AS n_baskets,
    AVG(CASE WHEN basket_sum = 0 THEN 1 ELSE 0 END)     AS share_zero_sum,
    QUANTILE_CONT(basket_sum, [0.01, 0.5, 0.99, 0.999]) AS basket_sum_q01_q50_q99_q999,
    MAX(basket_sum)                                     AS max_basket_sum,
    QUANTILE_CONT(n_items, [0.5, 0.99])                 AS items_q50_q99,
    MAX(n_items)                                        AS max_items
FROM b
""")

quantity = q("""
SELECT
    QUANTILE_CONT(product_quantity, [0.5, 0.99, 0.999, 0.9999])                  AS qty_q50_q99_q999_q9999,
    AVG(CASE WHEN product_quantity = 0 THEN 1 ELSE 0 END)                        AS share_zero_qty,
    AVG(CASE WHEN product_quantity <> FLOOR(product_quantity) THEN 1 ELSE 0 END) AS share_fractional_qty
FROM purchases
""")

display(gender)
display(baskets.T)
display(quantity.T)

* Gender is not recorded (`U`) for 46% of clients. It is kept as its own category, not imputed.
* Basket spend is strongly right skewed (median 282, 99th percentile 2,372, maximum 35,149; the currency is not given), so spend features will need a log scale or similar.
* 5.9% of item rows have quantity 0 and no quantity is fractional, although a grocer sells many weighed goods. `product_quantity` is therefore not used; the amount bought is measured by item rows or distinct products instead.

### Age

In [ ]:
ages_df = q("SELECT age, COUNT(*) AS n FROM clients WHERE age BETWEEN 0 AND 125 GROUP BY age ORDER BY age")
n_hidden = q("SELECT COUNT(*) AS n FROM clients WHERE age < 0 OR age > 125")["n"][0]
n_clients = q("SELECT COUNT(*) AS n FROM clients")["n"][0]

invalid = (ages_df["age"] < AGE_MIN) | (ages_df["age"] > AGE_MAX)
n_invalid = int(ages_df.loc[invalid, "n"].sum()) + n_hidden
pct_invalid = 100 * n_invalid / n_clients
spikes = ages_df[ages_df["age"] > AGE_MAX].nlargest(2, "n")["age"].sort_values().tolist()

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(ages_df.loc[~invalid, "age"], ages_df.loc[~invalid, "n"], width=0.9, color=P.BLUE,
       label=f"Kept ({AGE_MIN} to {AGE_MAX})")
ax.bar(ages_df.loc[invalid, "age"], ages_df.loc[invalid, "n"], width=0.9, color=P.ORANGE,
       label="Treated as missing")
for a in spikes:
    v = ages_df.loc[ages_df["age"] == a, "n"].iloc[0]
    ax.annotate(f"{a}", xy=(a, v), xytext=(0, 4), textcoords="offset points",
                ha="center", fontsize=9, color=P.INK_2)
ax.set_xlim(-2, 127)
ax.set_xlabel("Age stated on the loyalty card (years)")
ax.set_ylabel("Clients")
ax.yaxis.set_major_formatter(lambda v, _: f"{v/1000:.0f}K" if v >= 1000 else f"{v:.0f}")
ax.legend(loc="upper right")
P.titles(ax, f"{pct_invalid:.2f}% of clients have an implausible age, with placeholder spikes at {spikes[0]} and {spikes[1]}",
         f"Age distribution of all {n_clients:,} clients; {n_hidden:,} values below 0 or above 125 are not shown")
fig.tight_layout()
P.save(fig, "01_age_cleaning")
plt.show()

Ages outside 14 to 100 are treated as missing (0.44% of clients) and flagged, not imputed. The spikes at 115 and 119 look like system default values rather than real ages. The two peaks and the dip at age 76 to 77 (people born in 1942 and 1943) are consistent with the Russian population structure, which suggests the remaining ages are mostly genuine.

## 2. When was the decision made?

The data does not give the date of the SMS. The purchase history ends at 2019-03-18 23:40:03 and the competition README describes it as the history *before* the SMS campaign, so this moment is used as the decision moment (`CUTOFF`). Every model input must be known at or before it.

In [ ]:
check = q(f"""
SELECT
    (SELECT MAX(transaction_datetime) FROM purchases) = TIMESTAMP '{CUTOFF}' AS cutoff_is_history_end,
    (SELECT COUNT(*) FROM clients WHERE first_issue_date > TIMESTAMP '{CUTOFF}') AS cards_issued_after_cutoff
""")
assert check.loc[0, "cutoff_is_history_end"] and check.loc[0, "cards_issued_after_cutoff"] == 0
check

If the SMS had been sent before the history ends, the treatment group would already shop more in the last days of the history. The next two cells compare daily baskets per client in the two groups over the last three weeks.

In [ ]:
daily = q("""
WITH daily AS (
    SELECT t.treatment_flg, CAST(p.transaction_datetime AS DATE) AS day,
           COUNT(DISTINCT p.transaction_id) AS n_txn
    FROM purchases AS p
    JOIN train AS t USING (client_id)
    WHERE p.transaction_datetime >= TIMESTAMP '2019-02-25'
    GROUP BY 1, 2
),
group_size AS (SELECT treatment_flg, COUNT(*) AS n_clients FROM train GROUP BY 1)
SELECT
    d.day,
    MAX(CASE WHEN d.treatment_flg = 0 THEN d.n_txn * 1000.0 / g.n_clients END) AS txn_per_1000_control,
    MAX(CASE WHEN d.treatment_flg = 1 THEN d.n_txn * 1000.0 / g.n_clients END) AS txn_per_1000_treatment
FROM daily AS d
JOIN group_size AS g USING (treatment_flg)
GROUP BY d.day
ORDER BY d.day
""")
daily["ratio_treatment_to_control"] = daily["txn_per_1000_treatment"] / daily["txn_per_1000_control"]

In [ ]:
n_groups = q("SELECT treatment_flg, COUNT(*) AS n FROM train GROUP BY 1").set_index("treatment_flg")["n"]
n_c = daily["txn_per_1000_control"] * n_groups[0] / 1000    # daily baskets, control
n_t = daily["txn_per_1000_treatment"] * n_groups[1] / 1000  # daily baskets, treatment
diff_pct = (daily["ratio_treatment_to_control"] - 1) * 100
band_pct = 1.96 * np.sqrt(1 / n_c + 1 / n_t) * 100  # 95% range if only chance were at work (Poisson)

fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(daily["day"], -band_pct, band_pct, color=P.NEUTRAL_FILL, linewidth=0)
ax.axhline(0, color=P.AXIS, linewidth=1)
ax.plot(daily["day"], diff_pct, color=P.TREATMENT, marker="o", markersize=4,
        markeredgecolor=P.SURFACE, markeredgewidth=1.5)
ax.text(daily["day"].iloc[1], band_pct.iloc[1] + 0.25, "Range expected from chance alone (95%)",
        fontsize=9, color=P.INK_2)
ax.annotate("Purchase history ends\n(decision moment)", xy=(daily["day"].iloc[-1], diff_pct.iloc[-1]),
            xytext=(-10, 28), textcoords="offset points", ha="right", fontsize=9, color=P.INK_2,
            arrowprops=dict(arrowstyle="-", color=P.MUTED, linewidth=1))
ax.set_ylim(-4, 4)
ax.set_ylabel("Treatment vs control (%)")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:+.0f}%" if v else "0%")
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
P.titles(ax, "No sign that the SMS was sent before the purchase history ends",
         "Daily baskets per client in the treatment group compared with the control group, last 3 weeks")
fig.tight_layout()
P.save(fig, "01_sms_timing_check")
plt.show()

The daily difference stays inside the range expected from chance (about ±2%) and does not rise at the end. The test cannot rule out an SMS sent in the last one or two days with a small effect, and the band assumes baskets are independent, so the true chance range is slightly wider. Neither point changes the conclusion.

## 3. A leak: the first redemption date

`clients.first_redeem_date` is the date a client first spent loyalty points. The clients table was exported months after the campaign, so this column can contain dates after the decision moment.

In [ ]:
redeem = q(f"""
SELECT
    DATE_TRUNC('month', c.first_redeem_date)   AS month,
    c.first_redeem_date > TIMESTAMP '{CUTOFF}' AS after_cutoff,
    COUNT(*)                                   AS n
FROM train AS t
JOIN clients AS c USING (client_id)
WHERE c.first_redeem_date IS NOT NULL
GROUP BY 1, 2
ORDER BY 1, 2
""")
n_train = q("SELECT COUNT(*) AS n FROM train")["n"][0]
redeem["pct"] = redeem["n"] / n_train * 100

wide = redeem.pivot(index="month", columns="after_cutoff", values="pct").fillna(0)
before = wide[False] if False in wide else 0
after = wide[True] if True in wide else 0
x = wide.index + pd.Timedelta(days=15)  # centre each bar inside its month

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x, before, width=26, color=P.BLUE, edgecolor=P.SURFACE, linewidth=0.8,
       label="First redemption before the decision moment")
ax.bar(x, after, width=26, bottom=before, color=P.ORANGE, edgecolor=P.SURFACE, linewidth=0.8,
       label="First redemption after the decision moment")
cutoff_ts = pd.Timestamp(CUTOFF)
ax.axvline(cutoff_ts, color=P.INK_2, linewidth=1)
ax.text(cutoff_ts, ax.get_ylim()[1] * 0.97, "Decision moment ", ha="right", va="top", fontsize=9, color=P.INK_2)
ax.set_ylabel("Share of experiment clients (%)")
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.legend(loc="upper left", bbox_to_anchor=(0, -0.1), ncol=2)
P.titles(ax, f"{after.sum():.1f}% of clients first redeemed points only after the decision moment",
         f"Month of the first point redemption in the raw clients table, {n_train:,} experiment clients")
fig.tight_layout()
P.save(fig, "01_redeem_date_leak")
plt.show()

One in nine experiment clients has a first redemption date after the decision moment, information nobody had when deciding whom to text. The next cell checks whether the SMS itself changed this column.

In [ ]:
q(f"""
SELECT
    t.treatment_flg,
    COUNT(*) AS n_clients,
    AVG(CASE WHEN c.first_redeem_date IS NULL THEN 1 ELSE 0 END)                    AS share_never_redeemed,
    AVG(CASE WHEN c.first_redeem_date > TIMESTAMP '{CUTOFF}' THEN 1 ELSE 0 END)     AS share_first_redeem_after_cutoff,
    AVG(CASE WHEN c.first_redeem_date <= TIMESTAMP '{CUTOFF}' THEN 1 ELSE 0 END)    AS share_first_redeem_before_cutoff
FROM train AS t
JOIN clients AS c USING (client_id)
GROUP BY t.treatment_flg
ORDER BY t.treatment_flg
""")

After the decision moment, 11.49% of the treatment group redeemed points for the first time against 11.10% of the control group (about 2.8 standard errors), so the SMS seems to have pushed some clients to redeem. Before the decision moment the shares are 80.07% and 79.79% (about 1.5 standard errors, consistent with chance).

The raw column is therefore both future information and partly an outcome of the experiment. Used as a model input, it would leak the answer; used as a control variable in the A/B analysis, it would bias the effect estimate. **Fix:** redemption dates after the decision moment are treated as "not redeemed yet" (see `features.py`). After the fix the column is balanced between the groups.

Note that a standardized difference check alone would have missed this: the gap in missing values corresponds to a standardized difference of only about 0.02.

## 4. Was the experiment randomized?

In [ ]:
groups = q("SELECT treatment_flg, COUNT(*) AS n_clients FROM train GROUP BY 1 ORDER BY 1").set_index("treatment_flg")
chi2, p = srm_chisquare(groups.loc[1, "n_clients"], groups.loc[0, "n_clients"], EXPECTED_TREATMENT_SHARE)
print(f"Sample ratio mismatch test against a 50/50 design: chi2 = {chi2:.3f}, p = {p:.3f}")
groups

The group sizes match a 50/50 assignment (p = 0.86), so there is no sign of clients being lost or added unevenly. The next cell compares the two groups on all 54 features known before the SMS (built in `features.py`, documented in notebook 02).

In [ ]:
features = pd.read_parquet(FEATURES_PATH)

# Standardized mean difference (SMD) of every pre-decision feature between the two groups
exp = q("SELECT client_id, treatment_flg FROM train").merge(features, on="client_id")
X = pd.get_dummies(exp.drop(columns=["client_id", "treatment_flg"]), columns=["gender"], dtype=float)
by_group = X.groupby(exp["treatment_flg"])
mean, var = by_group.mean(), by_group.var()
smd = ((mean.loc[1] - mean.loc[0]) / np.sqrt((var.loc[1] + var.loc[0]) / 2)).sort_values(key=abs)
n_t, n_c = (exp["treatment_flg"] == 1).sum(), (exp["treatment_flg"] == 0).sum()
chance = 1.96 * np.sqrt(1 / n_t + 1 / n_c)  # 95% range of an SMD when the groups truly do not differ
n_outside = int((smd.abs() > chance).sum())

fig, ax = plt.subplots(figsize=(8, 11))
y = np.arange(len(smd))
ax.axvspan(-chance, chance, color=P.NEUTRAL_FILL, linewidth=0)
for xline in (-0.1, 0.1):
    ax.axvline(xline, color=P.AXIS, linewidth=1)
ax.scatter(smd.values, y, s=30, color=P.TREATMENT, edgecolor=P.SURFACE, linewidth=1.5, zorder=3)
ax.set_yticks(y, smd.index, fontsize=8)
ax.set_xlim(-0.13, 0.13)
ax.set_ylim(-1, len(smd) + 1)
ax.grid(axis="y", visible=False)
ax.grid(axis="x", visible=True)
ax.text(0.097, len(smd) + 0.3, "Usual imbalance threshold", ha="right", fontsize=8, color=P.INK_2)
ax.text(0, -0.9, "Chance range (95%)", ha="center", fontsize=8, color=P.INK_2)
ax.set_xlabel("Standardized difference, treatment minus control")
verdict = "are balanced" if smd.abs().max() < 0.1 else "are NOT balanced"
P.titles(ax, f"Treatment and control {verdict} on all {len(smd)} pre-decision features",
         f"Largest difference {smd.abs().max():.3f} SD; {n_outside} of {len(smd)} outside the chance range "
         f"(about {0.05 * len(smd):.0f} expected); {n_t + n_c:,} clients")
fig.tight_layout()
P.save(fig, "01_balance")
plt.show()

The largest difference is 0.010 standard deviations, a tenth of the usual 0.1 threshold, and 2 of 54 features fall outside the 95% chance range where about 3 would be expected by chance. On everything known before the SMS, the two groups look alike, which is what random assignment should produce.

## 5. Splitting the labelled data

The 200,039 labelled clients are split once, stratified by treatment and outcome: 30% is set aside as a test set that no model, tuning step or chart touches until the final policy evaluation in notebook 06. The remaining 70% is divided into 5 folds for cross fitting, so that every client gets a prediction from a model that never saw them.

In [ ]:
labeled = q("SELECT client_id, treatment_flg, target FROM train")
splits = make_splits(labeled)
assert splits.equals(make_splits(labeled)), "split is not reproducible"

SPLITS_PATH.parent.mkdir(parents=True, exist_ok=True)
splits.to_parquet(SPLITS_PATH, index=False)

part = np.where(splits["is_test"], "test", "fold_" + splits["fold"].astype(str))
check = splits.assign(part=part).groupby("part").agg(
    n_clients=("client_id", "size"),
    treatment_share=("treatment_flg", "mean"),
    target_rate=("target", "mean"),
)
check["share_of_all"] = check["n_clients"] / len(splits)
check.round(4)

Every part has the same treatment share (0.4998) and outcome rate (0.6199) as the full data, and the split is reproducible from the fixed seed.

## Conclusion

**Answer.** Yes. Assignment to the SMS group was random in practice: group sizes match the design and the groups are balanced on all 54 features known before the SMS. Causal statements about the SMS can therefore be made inside this experiment.

**What we learned**
* Purchases are item rows; spend must be taken once per basket.
* The decision moment is the end of the purchase history, 2019-03-18 23:40:03, and there is no sign of the SMS inside the history.
* `first_redeem_date` leaked information from after the SMS and was partly caused by it; it is cut at the decision moment.
* Ages outside 14 to 100 and the quantity column are unreliable and are handled explicitly.

**What did not work or stays open**
* The exact meaning of `target` (the time window and what counts as a purchase) is not documented anywhere we could reach.
* Approximate distinct counts in DuckDB were off by up to 17%, so only exact counts are reported.
* Age and gender may also have been updated after the campaign; this cannot be tested.

**Next step.** Notebook 04 estimates the effect of the SMS in this A/B test, with confidence intervals and variance reduction.